In [ ]:
#!pip install statsmodels
#!pip install tensorflow
#!pip install torch torchvision

In [ ]:
# Data manipulatie
import pandas as pd
import statsmodels.api as sm

# Machine Learning
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

# Neurale Netwerken
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Visualisatie
import matplotlib.pyplot as plt
import seaborn as sns

# Vaste seed voor reproduceerbaarheid
seed = 6

### Voorbereiden data

In [ ]:
df_train = pd.read_csv('../zorgdata/df_train.csv', index_col=False)
df_valid = pd.read_csv('../zorgdata/df_valid.csv', index_col=False)

In [ ]:
# Selecteer variabelen om mee te nemen in de modellen
excluded = ['ct_id', 'datum', 'rapportage', 'onrustscore', 'rapportage_clean', 
            'rapportage_len_chars', 'rapportage_clean_len_chars', 'rapportage_clean_len_words', 'discipline']

label = 'onrust'

X_train = df_train.drop(excluded + [label], axis=1)
y_train = df_train[label]

X_valid = df_valid.drop(excluded + [label], axis=1)
y_valid = df_valid[label]

X_train_c = X_train.assign(const=1)

### Logistische regressie

In [ ]:
# Train het logistische regressiemodel
lr_model = sm.Logit(y_train, X_train_c)
result = lr_model.fit()

# Bereken de initiële AIC
initial_aic = result.aic

# Herhaal backward elimination totdat de AIC niet meer verbetert
while True:
    max_p_value = result.pvalues.max()

    if max_p_value > 0.05:
        variable_to_remove = result.pvalues.idxmax()
        X_train_c = X_train_c.drop(variable_to_remove, axis=1)
        
        # Gebruik sm.Logit consistent
        lr_model = sm.Logit(y_train, X_train_c)
        result = lr_model.fit()
        
        # Bereken de nieuwe AIC
        new_aic = result.aic
        
        # Stop als de AIC niet meer verbetert
        if new_aic >= initial_aic:
            break
        else:
            initial_aic = new_aic
    else:
        # Als alle p-waarden onder de drempelwaarde liggen, stop dan de eliminatie
        break

# Bekijk de uiteindelijke samenvatting van het model
print(result.summary2())

In [ ]:
# Geselecteerde variabelen na backward elimination
selected_variables = X_train_c.columns.tolist()
X_valid_c = df_valid.assign(const=1)[selected_variables]

In [ ]:
# Voorspellingen maken op de validatie dataset

# Maak een nieuwe df met de voorspellingen
df_valid_pred = df_valid[['onrustscore', 'onrust', 'rapportage']].copy()

# En voeg de voorspellingen toe
df_valid_pred['pred_lr_prob'] = result.predict(X_valid_c)
df_valid_pred['pred_lr'] = (df_valid_pred['pred_lr_prob'] >= 0.5).astype(int)

### Random forest

In [ ]:
# # Hyperparameters die je wilt testen
# param_grid = {
#     'n_estimators': [100, 501],
#     'max_depth': [None, 10, 30],
#     'min_samples_split': [2, 10],
#     'min_samples_leaf': [1, 4]
# }

# # Initialiseren van de Grid Search met cross-validation
# grid_search = GridSearchCV(estimator=RandomForestClassifier(random_state=seed), 
#                            param_grid=param_grid, 
#                            cv=5, # Aantal folds voor crossvalidaion
#                            n_jobs=-1, # Gebruik alle beschikbare CPU-cores
#                            verbose=2, # Toon gedetailleerde voortgangsinformatie
#                            scoring='roc_auc') # AUC als prestatie-indicator

# # Voer de grid search uit met trainingsdata
# grid_search.fit(X_train, y_train)

# # Beste parameters en bijbehorende AUC-score
# print("Beste hyperparameters:", grid_search.best_params_)
# print("Beste AUC-score:", grid_search.best_score_)

# # Beste model
# best_rf_model = grid_search.best_estimator_

In [ ]:
# Random Forest Model
rf_model = RandomForestClassifier(n_estimators=501, random_state=seed, max_depth=None, min_samples_leaf=4, min_samples_split=2)

# Trainen van het model met trainingsdata
rf_model.fit(X_train, y_train)

In [ ]:
# Voorspellingen maken op de validatieset
df_valid_pred['pred_rf'] = rf_model.predict(X_valid)
df_valid_pred['pred_rf_prob'] = rf_model.predict_proba(X_valid)[:, 1]

In [ ]:
# Variable importance 
feature_importances = pd.DataFrame(rf_model.feature_importances_,
                                   index = X_train.columns,
                                   columns=['importance']).sort_values('importance', ascending=False)
print(feature_importances[0:10])

### Tensorflow NN

In [ ]:
# Normaliseren van de variabelen
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

In [ ]:
# Bouwen van het neuraal netwerk
tf_model = Sequential()
tf_model.add(Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)))
tf_model.add(Dense(32, activation='relu'))
tf_model.add(Dense(1, activation='sigmoid')) # Uitvoerlaag voor binaire classificatie

In [ ]:
# Compileren van het model
tf_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
# Train het model
history = tf_model.fit(X_train_scaled, y_train, epochs=10, batch_size=32, validation_data=(X_valid_scaled, y_valid))

In [ ]:
# Evaluatie van het model
loss, accuracy = tf_model.evaluate(X_valid_scaled, y_valid)
print(f'Validatie nauwkeurigheid: {accuracy:.2f}, Verlies: {loss:.2f}')

In [ ]:
# Bereken de voorspellingen voor de validatieset
df_valid_pred['pred_tf_prob'] = tf_model.predict(X_valid_scaled)
df_valid_pred['pred_tf'] = (df_valid_pred['pred_tf_prob'] > 0.5).astype("int32")

### PyTorch NN

In [ ]:
# Data voorbereiden
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)

# Omzetten van data naar PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
X_valid_tensor = torch.tensor(X_valid_scaled, dtype=torch.float32)
y_valid_tensor = torch.tensor(y_valid.values, dtype=torch.float32)

# Datasets en DataLoaders maken
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
valid_dataset = TensorDataset(X_valid_tensor, y_valid_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

In [ ]:
# Neuraal netwerk definieren
class NeuralNet(nn.Module):
    def __init__(self):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(X_train_scaled.shape[1], 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

# Model, verliesfunctie en optimizer initialiseren
pt_model = NeuralNet()
criterion = nn.BCELoss()
optimizer = optim.Adam(pt_model.parameters(), lr=0.001)

In [ ]:
# Trainen van het model
num_epochs = 10
for epoch in range(num_epochs):
    pt_model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = pt_model(X_batch)
        loss = criterion(y_pred.squeeze(), y_batch)
        loss.backward()
        optimizer.step()

    # Valideren van het model
    pt_model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for X_batch, y_batch in valid_loader:
            y_pred = pt_model(X_batch)
            predicted = (y_pred.squeeze() > 0.5).float()
            total += y_batch.size(0)
            correct += (predicted == y_batch).sum().item()

    accuracy = 100 * correct / total
    print(f'Epoch {epoch+1}, Verlies: {loss.item():.4f}, Nauwkeurigheid: {accuracy:.2f}%')

In [ ]:
# Berekenen van de voorspelde waarschijnlijkheden voor het Random Forest-model
pt_model.eval()  # Zet het model in evaluatiemodus
with torch.no_grad():
    df_valid_pred['pred_pt_prob'] = pt_model(X_valid_tensor).numpy().ravel()

# Converteer de voorspelde waarschijnlijkheden naar labels
df_valid_pred['pred_pt'] = (df_valid_pred['pred_pt_prob'] >= 0.5)

In [ ]:
def plot_kruistabel(data, model_name, subplot):
    kruistabel = pd.crosstab(data['onrust'], data[model_name], 
                             rownames=['Werkelijke Onrust'], colnames=['Voorspelde Onrust'])
    sns.heatmap(kruistabel, annot=True, fmt='d', cmap='Blues', ax=subplot)
    subplot.set_title(model_name)

# Maak een figuur en definieer subplots
fig, axs = plt.subplots(1, 4, figsize=(20, 5))  # 4 modellen naast elkaar

# Plot elke kruistabel in zijn eigen subplot
plot_kruistabel(df_valid_pred, 'pred_lr', axs[0])
plot_kruistabel(df_valid_pred, 'pred_rf', axs[1])
plot_kruistabel(df_valid_pred, 'pred_tf', axs[2])
plot_kruistabel(df_valid_pred, 'pred_pt', axs[3])

plt.tight_layout()
plt.show()

In [ ]:
# Bereken de ROC-curve
fpr_lr, tpr_lr, thresholds_lr = roc_curve(df_valid_pred['onrust'], df_valid_pred['pred_lr_prob'])
fpr_rf, tpr_rf, thresholds_rf = roc_curve(df_valid_pred['onrust'], df_valid_pred['pred_rf_prob'])
fpr_tf, tpr_tf, thresholds_tf = roc_curve(df_valid_pred['onrust'], df_valid_pred['pred_tf_prob'])
fpr_pt, tpr_pt, thresholds_pt = roc_curve(df_valid_pred['onrust'], df_valid_pred['pred_pt_prob'])

# Bereken de AUC (Area Under Curve)
roc_auc_lr = roc_auc_score(df_valid_pred['onrust'], df_valid_pred['pred_lr_prob'])
roc_auc_rf = roc_auc_score(df_valid_pred['onrust'], df_valid_pred['pred_rf_prob'])
roc_auc_tf = roc_auc_score(df_valid_pred['onrust'], df_valid_pred['pred_tf_prob'])
roc_auc_pt = roc_auc_score(df_valid_pred['onrust'], df_valid_pred['pred_pt_prob'])

plt.figure(figsize=(10, 4))

# ROC Curves
plt.plot(fpr_lr, tpr_lr, label='Logistisch Regressie (AUC = {:.2f})'.format(roc_auc_lr), color='blue', linestyle='-')
plt.plot(fpr_rf, tpr_rf, label='Random Forest (AUC = {:.2f})'.format(roc_auc_rf), color='green', linestyle='--')
plt.plot(fpr_tf, tpr_tf, label='Neuraal Netwerk (TensorFlow) (AUC = {:.2f})'.format(roc_auc_tf), color='red', linestyle='-.')
plt.plot(fpr_pt, tpr_pt, label='Neuraal Netwerk (PyTorch) (AUC = {:.2f})'.format(roc_auc_pt), color='purple', linestyle=':')

# Baseline
plt.plot([0, 1], [0, 1], 'k--')

# Assen en Titel
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Vergelijking', fontsize=14)

# Legenda
plt.legend(loc='lower right', fontsize=10)

# Grid
plt.grid(True)

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])

plt.show()